# Debugging toolkit for research notebooks

How to find what is wrong in a notebook you did not write, quickly and systematically.
Built for the "here is a notebook, tell us what you would investigate" interview format.

**What's in here**
- reproduce first: seeds, restart-and-run-all, shape tracing with `.pipe`
- assertions as documentation; `pd.testing` and `np.testing`
- bisecting a pipeline: recompute two ways, diff two frames
- inspecting intermediate objects around a known bad timestamp
- automatic leakage tests
- silent-failure detectors
- warnings as errors, copy-on-write, `%debug`, re-raising
- minimal reproducible example
- reading tracebacks and the pandas errors you will meet
- the six checks when someone says "R² is 0.99"
- quick reference

In [1]:
import warnings
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

raw = pd.read_csv("../data/hourly_power_raw.csv")
clean = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
raw.shape, clean.shape

((17457, 7), (17520, 6))

## 1. Reproduce first

Before diagnosing anything: restart the kernel and run all cells top to bottom, fix every
random seed, and record the shape of every intermediate object. Hidden state (a cell run
twice, a variable overwritten out of order) explains a surprising share of "bugs".

A `trace` helper you drop into `.pipe(...)` chains prints the vital signs and returns the
frame unchanged, so it can sit inside a pipeline without altering it.

In [2]:
def trace(df, label=""):
    """Print vital signs and return df unchanged; use inside .pipe()."""
    idx = df.index
    print(f"[{label:>12s}] shape={df.shape}  index={type(idx).__name__:<14s} "
          f"unique={idx.is_unique}  monotonic={idx.is_monotonic_increasing}  "
          f"nan={int(df.isna().sum().sum())}  dup_rows={int(df.duplicated().sum())}")
    return df

out = (raw
       .pipe(trace, "raw")
       .assign(time=lambda d: pd.to_datetime(d["time"], utc=True))
       .drop_duplicates()
       .pipe(trace, "dedup")
       .set_index("time").sort_index()
       .pipe(trace, "indexed"))

[         raw] shape=(17457, 7)  index=RangeIndex     unique=True  monotonic=True  nan=149  dup_rows=15
[       dedup] shape=(17442, 7)  index=Index          unique=True  monotonic=True  nan=149  dup_rows=0
[     indexed] shape=(17442, 6)  index=DatetimeIndex  unique=True  monotonic=True  nan=149  dup_rows=0


**Pitfall:** here `drop_duplicates()` was enough because the 15 duplicates were exact
copies. If a duplicated timestamp carries *different* values (a corrected reading sent
twice), exact-row dedup leaves both and the index stays non-unique. Always run the key
check as well.

In [3]:
raw2 = raw.copy()
dup_pos = raw2.index[raw2.duplicated(keep=False)][:2]
raw2.loc[dup_pos[0], "temp_c"] = 99.0                 # one copy now disagrees
out2 = (raw2.assign(time=lambda d: pd.to_datetime(d["time"], utc=True))
            .drop_duplicates().set_index("time").sort_index().pipe(trace, "row-dedup"))
out2 = out2[~out2.index.duplicated(keep="last")].pipe(trace, "key-dedup")

[   row-dedup] shape=(17443, 6)  index=DatetimeIndex  unique=False  monotonic=True  nan=149  dup_rows=0
[   key-dedup] shape=(17442, 6)  index=DatetimeIndex  unique=True  monotonic=True  nan=149  dup_rows=0


## 2. Assertions as documentation

An `assert` is a one-line statement of what you believe about the data at that point.
When it fails you learn something immediately instead of three cells later. Row counts
before and after a merge are the most valuable one.

In [4]:
assert out.index.is_unique
assert out.index.is_monotonic_increasing
assert out.index.tz is not None, "timestamps must be tz-aware"

meters = pd.read_csv("../data/meters.csv")
readings = pd.read_csv("../data/meter_readings_daily.csv")
n_before = len(readings)
merged = readings.merge(meters, on="meter_id", how="left", validate="many_to_one")
assert len(merged) == n_before, f"merge changed row count {n_before} -> {len(merged)}"
print("all assertions passed; unmatched readings:", merged["region"].isna().sum())

all assertions passed; unmatched readings: 200


In [5]:
# pd.testing / np.testing: compare with tolerance and useful diffs
a = clean.set_index("time")["consumption_mwh"]
b = out["consumption_mwh"].reindex(a.index)
try:
    pd.testing.assert_series_equal(a, b)
except AssertionError as e:
    print("assert_series_equal:", str(e).splitlines()[0])
print("max abs diff where both present:", (a - b).abs().max())
np.testing.assert_allclose(a.dropna().values[:5], b.dropna().values[:5], atol=1e-9)
print("first five values allclose: ok")

assert_series_equal: Series are different
max abs diff where both present: 0.0
first five values allclose: ok


## 3. Bisecting a pipeline

When a result is wrong, halve the problem: recompute the suspicious quantity a second,
independent way on a handful of rows and compare. A five-row Python loop is slow but
hard to get wrong, which makes it a good oracle for a vectorised expression.

In [6]:
s = a.copy()
vec = s.shift(1).rolling(24).mean()          # the feature under suspicion

def loop_feature(series, t):
    """Mean of the 24 values strictly before t, or NaN."""
    pos = series.index.get_loc(t)
    if pos < 24:
        return np.nan
    return series.iloc[pos - 24:pos].mean()

check = pd.DataFrame({"vectorised": vec.iloc[24:29],
                      "loop": [loop_feature(s, t) for t in s.index[24:29]]})
check["match"] = np.isclose(check["vectorised"], check["loop"])
check

,vectorised,loop,match
time,,,
2022-01-02 00:00:00+00:00,30587.241667,30587.241667,True
2022-01-02 01:00:00+00:00,30584.195833,30584.195833,True
2022-01-02 02:00:00+00:00,30569.558333,30569.558333,True
2022-01-02 03:00:00+00:00,30551.658333,30551.658333,True
2022-01-02 04:00:00+00:00,30580.829167,30580.829167,True


In [7]:
# diffing two frames: df.compare, indicator merges, index set differences
x = clean.set_index("time").head(6)
y = x.copy(); y.iloc[2, 0] += 100; y.iloc[4, 2] = np.nan
print(x.compare(y))

left_only = out.index.symmetric_difference(a.index)
print("timestamps in one frame but not the other:", len(left_only), "->", left_only[:2].tolist())

m = readings.merge(meters[["meter_id"]], on="meter_id", how="outer", indicator=True)
print(m["_merge"].value_counts().to_dict())

                          consumption_mwh          wind_ms      
                                     self    other    self other
time                                                            
2022-01-01 02:00:00+00:00         26229.4  26329.4     NaN   NaN
2022-01-01 04:00:00+00:00             NaN      NaN    7.36   NaN
timestamps in one frame but not the other: 78 -> [Timestamp('2022-01-07 10:00:00+0000', tz='UTC'), Timestamp('2022-01-17 14:00:00+0000', tz='UTC')]
{'both': 107303, 'left_only': 200, 'right_only': 0}


## 4. Inspecting intermediate objects

Cheap commands that reveal most problems. Run them on every intermediate frame, not just
the input. Slicing a few hours around a known bad timestamp shows the shape of the
problem (gap, duplicate, step change) better than any aggregate.

In [8]:
print(raw.dtypes.value_counts().to_dict())
print(raw.describe(include="all").loc[["count", "unique", "top", "min", "max"]].T)

{dtype('float64'): 4, dtype('O'): 3}


                   count unique                  top      min      max
time               17457  17442  2022-02-04 07:00:00      NaN      NaN
consumption_mwh  17457.0    NaN                  NaN  18092.9  40824.9
temp_c           17308.0    NaN                  NaN   -999.0    27.74
wind_ms          17457.0    NaN                  NaN      0.0    15.99
solar_wm2        17457.0    NaN                  NaN      0.0    794.6
price_eur_mwh      17457   9932              missing      NaN      NaN
region             17457      1                   GB      NaN      NaN


In [9]:
t = pd.Timestamp("2022-03-27 12:00", tz="UTC")      # a day we suspect is missing
window = out.loc[t - pd.Timedelta("30h"): t + pd.Timedelta("30h"), ["consumption_mwh", "temp_c"]]
print("rows in a 60h window (expect 61):", len(window))
print("gap positions:", window.index.to_series().diff().pipe(lambda d: d[d > pd.Timedelta("1h")]).to_dict())

rows in a 60h window (expect 61): 37
gap positions: {Timestamp('2022-03-28 00:00:00+0000', tz='UTC'): Timedelta('1 days 01:00:00')}


## 5. Leakage tests you can run automatically

A leaked feature usually reveals itself in one of five ways. Build them into a function
and run it whenever a result looks too good.

**Interview check:** "How would you *prove* a feature is leaking rather than just being
good?" Two tests. (1) Beat-the-persistence test: on a one-hour horizon nothing legitimate
should beat "consumption right now" by a wide margin. (2) Lead-lag test: correlate the
feature with consumption shifted by k = −2..+2 hours. A legitimate feature correlates
best with the present or the past (k ≤ 0); a feature whose best correlation is with the
*future* (k > 0) contains information from after the decision time.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

d = clean.set_index("time")[["consumption_mwh", "temp_c"]].copy()
d["target"] = d["consumption_mwh"].shift(-1)                       # next-hour consumption
d["honest"] = d["consumption_mwh"].shift(1).rolling(24).mean()     # available at t
d["leaked"] = d["consumption_mwh"].shift(-1).rolling(3, center=True).mean()  # uses t+1 and t+2
d = d.dropna()
split = int(len(d) * 0.8)
tr, te = d.iloc[:split], d.iloc[split:]

def leakage_report(tr, te, feats, target="target"):
    rows = {}
    for f in feats:
        m = LinearRegression().fit(tr[[f]], tr[target])
        r2_test = r2_score(te[target], m.predict(te[[f]]))
        r2_train = r2_score(tr[target], m.predict(tr[[f]]))
        corr_now = np.corrcoef(te[f], te[target])[0, 1]
        ks = range(-2, 3)
        xcorr = {k: te[f].corr(te["consumption_mwh"].shift(-k)) for k in ks}   # k>0: future consumption
        best_k = max(xcorr, key=xcorr.get)
        rng = np.random.default_rng(0)
        r2_shuffled = r2_score(te[target], LinearRegression().fit(tr[[f]], rng.permutation(tr[target].values)).predict(te[[f]]))
        rows[f] = {"r2_train": r2_train, "r2_test": r2_test, "corr_with_target": corr_now,
                   "best_lead_k": best_k, "r2_after_target_shuffle": r2_shuffled}
    return pd.DataFrame(rows).T.round(3)

leakage_report(tr, te, ["honest", "leaked", "consumption_mwh"])

,r2_train,r2_test,corr_with_target,best_lead_k,r2_after_target_shuffle
honest,0.251,0.140,0.374,-2.0,-0.018
leaked,0.992,0.991,0.995,1.0,0.002
consumption_mwh,0.878,0.855,0.925,0.0,-0.003


Reading the table: `consumption_mwh` at time *t* is the persistence benchmark and its
best lead is k = 0. The `leaked` feature beats persistence by a margin no honest feature
could on a one-hour horizon, and its best lead is k = +1: it is most correlated with
consumption *after* the decision time. The shuffle column is the floor: every feature
scores about zero against a shuffled target, so the fit itself is not leaking.

In [11]:
# Permutation test of one suspicious feature inside a multi-feature model
feats = ["honest", "leaked", "temp_c"]
model = LinearRegression().fit(tr[feats], tr["target"])
base = r2_score(te["target"], model.predict(te[feats]))
rng = np.random.default_rng(1)
drops = {}
for f in feats:
    tmp = te[feats].copy()
    tmp[f] = rng.permutation(tmp[f].values)
    drops[f] = base - r2_score(te["target"], model.predict(tmp))
print("test R² with all features:", round(base, 3))
print("R² drop when each feature is permuted:", {k: round(v, 3) for k, v in drops.items()})

test R² with all features: 0.991
R² drop when each feature is permuted: {'honest': 0.001, 'leaked': 2.015, 'temp_c': 0.0}


## 6. Silent-failure detectors

Things that do not raise but are wrong. Wrap them in one function and run it on any
frame you are about to model with.

In [12]:
def silent_failures(df):
    issues = []
    for c in df.columns:
        s = df[c]
        if s.dtype == object and pd.to_numeric(s, errors="coerce").notna().mean() > 0.9:
            issues.append(f"'{c}' is object dtype but ~numeric (dirty strings?)")
        if s.isna().all():
            issues.append(f"'{c}' is all-NaN (failed merge or wrong column name?)")
        if s.nunique(dropna=False) == 1:
            issues.append(f"'{c}' is constant ({s.iloc[0]!r})")
        if pd.api.types.is_numeric_dtype(s) and (s == -999).any():
            issues.append(f"'{c}' contains -999 sentinels")
    if df.duplicated().sum():
        issues.append(f"{df.duplicated().sum()} duplicated rows")
    tzs = {str(df[c].dt.tz) for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])}
    if len(tzs) > 1:
        issues.append(f"mixed timezones across datetime columns: {tzs}")
    return issues or ["no silent failures detected"]

for line in silent_failures(raw):
    print("-", line)

- 'temp_c' contains -999 sentinels
- 'price_eur_mwh' is object dtype but ~numeric (dirty strings?)
- 'region' is constant ('GB')
- 15 duplicated rows


In [13]:
# inplace returns None; a suspiciously round number; the SettingWithCopy no-op
x = raw.copy()
result = x.dropna(inplace=True)
print("dropna(inplace=True) returned:", result)

metric = 0.9999999
print("round-number smell:", metric, "-> ask what was compared to what")

SENTINEL = -12345.0                        # a value that cannot occur naturally (0.0 can!)
sub = x[x["temp_c"] < -5]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sub["temp_c"] = SENTINEL                # writes to a copy (or warns); x is unchanged
print("x changed by the chained write?", (x["temp_c"] == SENTINEL).any())

dropna(inplace=True) returned: None
round-number smell: 0.9999999 -> ask what was compared to what
x changed by the chained write? False


## 7. Turn warnings into errors, and other kernel-level tools

Pandas warnings are easy to scroll past. During debugging make them fatal so the stack
trace points at the offending line. Copy-on-write mode removes the copy/view ambiguity
entirely and is a good experiment when a chained assignment is suspected.

In [14]:
with warnings.catch_warnings():
    warnings.simplefilter("error")
    try:
        pd.to_datetime(pd.Series(["01/02/2023", "13/02/2023"]))   # day-first ambiguity warns -> now raises
    except Exception as e:
        print(type(e).__name__, "->", str(e)[:70])

pd.options.mode.copy_on_write = True
y = raw.copy(); sub = y[y["temp_c"] < -5]; sub.loc[:, "temp_c"] = SENTINEL
print("with copy_on_write, parent untouched:", not (y["temp_c"] == SENTINEL).any())
pd.options.mode.copy_on_write = False

ValueError -> time data "13/02/2023" doesn't match format "%m/%d/%Y", at position 1.
with copy_on_write, parent untouched: True


Other tools worth knowing, not runnable in a batch build but daily in a live notebook:

- `%debug` (or `import pdb; pdb.pm()`) right after an exception opens the post-mortem
  debugger at the failing frame: inspect `df.shape`, `key`, whatever was in scope.
- `%xmode Plain` shortens tracebacks; `%xmode Verbose` shows local variables per frame.
- `try: ... except Exception: ...; raise` — log context, then re-raise. Never
  `except: pass` in research code; that is how mock 11's negative-price days vanished.
- `logging.info` with timestamps instead of bare `print` when a loop runs for minutes.

## 8. Minimal reproducible example

Shrink until the bug still shows, then fix on the small case and expand. Twenty rows that
reproduce a misalignment are easier to reason about than 17,520.

In [15]:
small = clean.set_index("time")["consumption_mwh"].iloc[:8]
X = small.to_frame("lag0")
y = small.shift(-1)
Xd, yd = X.dropna(), y.dropna()
print("naive approach lengths:", len(Xd), len(yd))
print("positional pairing (wrong):")
print(pd.DataFrame({"X_time": Xd.index[:3], "y_time": yd.index[:3]}))
frame = pd.concat([X, y.rename("target")], axis=1).dropna()
print("aligned frame:", frame.shape, "-> rows share one timestamp each")

naive approach lengths: 8 7
positional pairing (wrong):
                     X_time                    y_time
0 2022-01-01 00:00:00+00:00 2022-01-01 00:00:00+00:00
1 2022-01-01 01:00:00+00:00 2022-01-01 01:00:00+00:00
2 2022-01-01 02:00:00+00:00 2022-01-01 02:00:00+00:00
aligned frame: (7, 2) -> rows share one timestamp each


## 9. Reading tracebacks and the pandas errors you will meet

Read a traceback from the bottom: the last line is the error, the frame just above the
library internals is *your* line. The messages below are the ones that appear most in
research code, with their usual cause.

In [16]:
demos = {}
s1 = pd.Series([1, 2], index=["a", "b"]); s2 = pd.Series([1, 2], index=["b", "c"])
try: s1 == s2
except Exception as e: demos["Series with different index compared"] = e
dup = pd.Series([1, 2, 3], index=[0, 0, 1])
try: dup.reindex([0, 1, 2])
except Exception as e: demos["reindex with duplicate labels"] = e
try: raw["time"].dt.hour
except Exception as e: demos["strings, not datetimes"] = e
try:
    if pd.Series([True, False]): pass
except Exception as e: demos["Series in boolean context"] = e
try: readings.merge(meters, on="meter_id", validate="one_to_one")
except Exception as e: demos["merge validate"] = e
try: pd.Timestamp("2023-01-01") < pd.Timestamp("2023-01-01", tz="UTC")
except Exception as e: demos["tz-naive vs tz-aware"] = e
try: clean.merge(clean, on="time")["temp_c"]
except Exception as e: demos["KeyError after merge with suffixes"] = e

for cause, err in demos.items():
    print(f"{cause:40s} -> {type(err).__name__}: {str(err)[:60]}")

Series with different index compared     -> ValueError: Can only compare identically-labeled Series objects
reindex with duplicate labels            -> ValueError: cannot reindex on an axis with duplicate labels
strings, not datetimes                   -> AttributeError: Can only use .dt accessor with datetimelike values
Series in boolean context                -> ValueError: The truth value of a Series is ambiguous. Use a.empty, a.boo
merge validate                           -> MergeError: Merge keys are not unique in left dataset; not a one-to-one 
tz-naive vs tz-aware                     -> TypeError: Cannot compare tz-naive and tz-aware timestamps
KeyError after merge with suffixes       -> KeyError: 'temp_c'


## 10. "R² is 0.99 — is that fine?" The six checks

Run in this order; each takes under a minute. Stop when one fails and say why.

In [17]:
d = clean.set_index("time")[["consumption_mwh", "temp_c"]].copy()
d["target"] = d["consumption_mwh"].shift(-24)
d["lag24"] = d["consumption_mwh"]
d["lag168"] = d["consumption_mwh"].shift(144)
d["roll24"] = d["consumption_mwh"].rolling(24).mean()          # ends at t: fine for a 24h horizon
d["roll_c"] = d["consumption_mwh"].shift(-25).rolling(3).mean()  # "smoothed load": hours t+23..t+25, centred on the target
d = d.dropna()
feats = ["lag24", "lag168", "roll24", "roll_c", "temp_c"]
split = int(len(d) * 0.8); tr, te = d.iloc[:split], d.iloc[split:]
m = LinearRegression().fit(tr[feats], tr["target"])
print("reported test R²:", round(r2_score(te["target"], m.predict(te[feats])), 4))

reported test R²: 0.9916


In [18]:
# 1. What is one row, and what is the target?  2. Is the split chronological?
print("1. target = consumption 24h ahead; rows are hours; split at", te.index[0], "(chronological)")

# 3. Beat the naive baseline by how much?
naive = te["lag24"]
print("3. naive same-hour-yesterday R²:", round(r2_score(te["target"], naive), 3), "vs model", round(r2_score(te["target"], m.predict(te[feats])), 3))

# 4. Coefficients: anything near 1 on a single feature, or a pair that cancels?
print("4. coefficients:", pd.Series(m.coef_, index=feats).round(3).to_dict())

# 5. How was each feature computed? Correlation with the *future* target is the tell.
print("5. corr(feature, target):", te[feats].corrwith(te["target"]).round(3).to_dict())

# 6. Drop the suspect and refit
feats_ok = [f for f in feats if f != "roll_c"]
m2 = LinearRegression().fit(tr[feats_ok], tr["target"])
print("6. R² without the suspect feature:", round(r2_score(te["target"], m2.predict(te[feats_ok])), 3))

1. target = consumption 24h ahead; rows are hours; split at 2023-08-08 08:00:00+00:00 (chronological)
3. naive same-hour-yesterday R²: 0.828 vs model 0.992
4. coefficients: {'lag24': 0.048, 'lag168': 0.021, 'roll24': -0.069, 'roll_c': 0.983, 'temp_c': -8.308}
5. corr(feature, target): {'lag24': 0.914, 'lag168': 0.936, 'roll24': 0.281, 'roll_c': 0.995, 'temp_c': -0.0}
6. R² without the suspect feature: 0.898


The verdict: an R² that collapses when one feature is removed, with that feature carrying
the largest coefficient and the highest correlation with the target, is a leak until
proven otherwise. `roll_c` is a 3-hour mean centred on the target hour itself (a
"smoothed load" someone centred without thinking about time). Say that out loud, then read the feature's definition in the code.

## Quick reference

| Situation | Do this |
|---|---|
| any surprise | restart kernel, run all, fix seeds |
| pipeline of transforms | `.pipe(trace, "label")` between steps |
| merge | row count before/after, `validate=`, `indicator=True` |
| "is this feature right?" | recompute 5 rows by loop, `np.isclose` |
| two frames should match | `df.compare`, `pd.testing.assert_frame_equal(atol=)` |
| known bad timestamp | `df.loc[t - 3h : t + 3h]`, `diff()` of the index |
| result too good | six checks: row/target, split, baseline, coefficients, feature corr, drop-and-refit |
| leaked feature suspected | target shuffle → R²≈0; permute feature → drop; corr with t+1 target |
| warnings scroll past | `warnings.simplefilter("error")` |
| copy/view confusion | `pd.options.mode.copy_on_write = True` |
| exception | `%debug`, read the traceback bottom-up |
| big data, small bug | shrink to 20 rows that still reproduce |